# J2 après-midi — Transformer une table propre en informations utiles
**13h30–17h00 · · Programmation en Python · Mastère 1 Data & IA**

Passer de lignes de ventes à des indicateurs, réunir des sources et changer la forme d'une table.

Les démonstrations portent sur une **librairie fictive** ; les exercices et le TP portent sur **Online Retail II**.

**Objectifs :**

## 0. Démarrage
Dans Colab, ouvrez ce notebook avec **Fichier → Importer un notebook**.
Dans le volet **Fichiers** à gauche, importez les **quatre CSV** fournis, sans les renommer :
`sales_analysis.csv`, `products.csv`, `sales_part_1.csv`, `sales_part_2.csv`.

Exécutez les cellules dans l'ordre avec **Maj + Entrée**. Complétez les cellules `# TODO`.
Après une réinitialisation de la session Colab, réimportez les CSV et relancez les cellules déjà complétées.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)

In [6]:
# petite vérification que les fichiers requis soient bien présents
# si ça plante -> revoir l'import de vos .csv (mettez les au même niveau que votre notebook / racine du Colab)

required_files = [
    "sales_analysis.csv", "products.csv",
    "sales_part_1.csv", "sales_part_2.csv",
]
missing_files = [name for name in required_files if not Path(name).exists()]

assert not missing_files, f"Importez ces CSV dans Colab/Racine de votre projet : {missing_files}"

### Notre table de ventes
Une ligne représente une **ligne de facture** (un produit vendu). Une facture peut donc apparaître plusieurs fois.

| Colonne | Sens |
|---|---|
| `invoice_no` | Identifiant de facture |
| `stock_code` | Code produit |
| `description` | Description de cette ligne |
| `quantity` | Quantité vendue |
| `invoice_date` | Date et heure |
| `unit_price_gbp` | Prix unitaire en livres sterling |
| `customer_id` | Identifiant client, parfois absent |
| `country` | Pays |
| `market` | Marché pédagogique : UK, Europe ou Other |

Période : **du 1er janvier au 30 avril 2010 inclus**, sans échantillonnage.
Les ventes à quantité ou prix non positifs et les doublons ont déjà été écartés.
Le « CA » désigne ici le montant des ventes conservées, sans déduction des retours.
Un client non identifié n'empêche pas de calculer ce montant.

Source : [Chen, D. (2012), Online Retail II, UCI](https://archive.ics.uci.edu/dataset/502/online+retail+ii),
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
`market` et `product_family` sont des **métadonnées pédagogiques ajoutées**, pas des colonnes UCI.

In [7]:
# anticiper au chargement que certaines valeurs "numériques" nont en fait pas de "valeur" numérique ;)
identifier_types = {
    "invoice_no": "string",
    "stock_code": "string",
    "customer_id": "string",
}

sales_source = pd.read_csv(
    "sales_analysis.csv",
    dtype=identifier_types,
    parse_dates=["invoice_date"],
)

In [8]:
sales = sales_source.copy()

sales

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market
0,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346,United Kingdom,UK
1,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346,United Kingdom,UK
2,493413,21723,ALPHABET HEARTS STICKER SHEET,1,2010-01-04 09:54:00,0.85,<NA>,United Kingdom,UK
3,493413,21724,PANDA AND BUNNIES STICKER SHEET,1,2010-01-04 09:54:00,0.85,<NA>,United Kingdom,UK
4,493413,84578,ELEPHANT TOY WITH BLUE T-SHIRT,1,2010-01-04 09:54:00,3.75,<NA>,United Kingdom,UK
...,...,...,...,...,...,...,...,...,...
29473,496311,84029E,RED WOOLLY HOTTIE WHITE HEART.,2,2010-01-31 12:12:00,3.75,13190,United Kingdom,UK
29474,496311,85123A,WHITE HANGING HEART T-LIGHT HOLDER,4,2010-01-31 12:12:00,2.95,13190,United Kingdom,UK
29475,496313,16011,ANIMAL STICKERS,24,2010-01-31 12:15:00,0.21,15013,United Kingdom,UK
29476,496313,16012,FOOD/DRINK SPUNGE STICKERS,24,2010-01-31 12:15:00,0.21,15013,United Kingdom,UK


## 1. Des lignes aux indicateurs · 13h30–14h20
**Question : une ligne représente une transaction. Comment répondre à une question portant sur plusieurs lignes ?**

1. **Split** : former des groupes selon une ou plusieurs colonnes.
2. **Aggregate** : calculer une valeur par groupe (`sum`, `mean`, `nunique`…).
3. **Combine** : réunir les résultats dans une nouvelle table.

Principe et démonstration : **10–12 min**. Exercice A : **25 min**, puis correction et questions.

### Démonstration — Une librairie, trois magasins
Les huit lignes ci-dessous resteront notre seul univers de démonstration.
Une commande peut contenir plusieurs livres : compter les lignes ne compte donc pas les commandes.

In [9]:
bookstore_sales = pd.DataFrame(
    [
        ["J01", "B1", "Paris", "janvier", 2, 20],
        ["J01", "B2", "Paris", "janvier", 1, 15],
        ["J02", "B1", "Lyon", "janvier", 3, 20],
        ["J03", "B3", "Lille", "janvier", 2, 12],
        ["F01", "B2", "Lyon", "février", 4, 15],
        ["F02", "B3", "Paris", "février", 1, 12],
        ["F03", "B1", "Lille", "février", 2, 22],
        ["F03", "B3", "Lille", "février", 3, 12],
    ],
    columns=["order_id", "book_id", "store", "month", "quantity", "unit_price"],
)
bookstore_sales

,order_id,book_id,store,month,quantity,unit_price
0,J01,B1,Paris,janvier,2,20
1,J01,B2,Paris,janvier,1,15
2,J02,B1,Lyon,janvier,3,20
3,J03,B3,Lille,janvier,2,12
4,F01,B2,Lyon,février,4,15
5,F02,B3,Paris,février,1,12
6,F03,B1,Lille,février,2,22
7,F03,B3,Lille,février,3,12


**Combien de livres avons-nous vendus dans chaque magasin ?**
Commençons par voir les lignes qui appartiennent au groupe Paris.

In [10]:
bookstore_sales.loc[bookstore_sales["store"] == "Paris"]

,order_id,book_id,store,month,quantity,unit_price
0,J01,B1,Paris,janvier,2,20
1,J01,B2,Paris,janvier,1,15
5,F02,B3,Paris,février,1,12


In [12]:
bookstore_sales.groupby("store")["quantity"].sum()                   # [["quantity", "unit_price"]]

store
Lille    7
Lyon     7
Paris    4
Name: quantity, dtype: int64

`store` est devenu l'index du résultat. `reset_index()` le remet dans une colonne ordinaire.

In [13]:
bookstore_sales.groupby("store")["quantity"].sum().reset_index()

,store,quantity
0,Lille,7
1,Lyon,7
2,Paris,4


In [14]:
# mais si je veux plusieurs colonnes il se passe quoi ?? 🤔
bookstore_sales.groupby("store").sum()

C:\Users\hp\AppData\Local\Temp\ipykernel_11040\2120897592.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  bookstore_sales.groupby("store").sum()


,quantity,unit_price
store,,
Lille,7,46
Lyon,7,35
Paris,4,47


**Plusieurs indicateurs en une agrégation**
`agg` associe une colonne à une opération. `nunique` compte les valeurs distinctes ; par défaut, il ignore les valeurs manquantes.
Le prix moyen ci-dessous est une moyenne des prix des lignes, sans pondération par les quantités.

In [15]:
#.agg permet de faire des agrégations différentes selon les colonnes
bookstore_sales.groupby("store").agg({
    "quantity": "sum",
    "unit_price": "mean",
    "order_id": "nunique",
})

,quantity,unit_price,order_id
store,,,
Lille,7,15.333333,2
Lyon,7,17.500000,2
Paris,4,15.666667,2


### Exercice A — Où se concentre réellement l'activité ?
**25 minutes · Online Retail · table `sales`**

La direction veut comparer les pays et repérer les principaux clients.
Chaque calcul doit porter sur les bonnes lignes et sur la bonne unité : ligne de facture, facture ou client.

**A1. Montant des lignes.** Créez `line_total_gbp` dans `sales` avec `quantity * unit_price_gbp`. Affichez les cinq premières valeurs.

In [ ]:
bookst

In [19]:
# TODO A1
sales["line_total_gbp"] = sales["quantity"] * sales["unit_price_gbp"]
sales.head(5)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market,line_total_gbp
0,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346,United Kingdom,UK,22.50
1,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346,United Kingdom,UK,22.50
2,493413,21723,ALPHABET HEARTS STICKER SHEET,1,2010-01-04 09:54:00,0.85,<NA>,United Kingdom,UK,0.85
3,493413,21724,PANDA AND BUNNIES STICKER SHEET,1,2010-01-04 09:54:00,0.85,<NA>,United Kingdom,UK,0.85
4,493413,84578,ELEPHANT TOY WITH BLUE T-SHIRT,1,2010-01-04 09:54:00,3.75,<NA>,United Kingdom,UK,3.75


**A2. Indicateurs pays.** Construisez `country_kpis` : pour chaque pays, le CA total, le nombre de factures distinctes et le nombre de clients identifiés distincts. Une facture comportant dix lignes ne compte qu'une fois.

In [25]:
# TODO A2
country_kpis = sales.groupby("country").agg({
    "line_total_gbp": "sum",
    "invoice_no": "nunique",
    "customer_id": "nunique",
})

country_kpis.sort_values(by="line_total_gbp", ascending=False)

,line_total_gbp,invoice_no,customer_id
country,,,
United Kingdom,502446.852,970,642
EIRE,64232.050,31,5
Netherlands,29230.770,6,3
Germany,10806.370,20,12
France,8145.370,11,8
...,...,...,...
Portugal,117.720,1,1
Cyprus,76.520,1,1
Italy,39.670,1,1


Remettez `country` dans une colonne. Nommez les colonnes `revenue_gbp`, `invoice_count`, `identified_customer_count`.

In [27]:
# TODO A2
country_kpis = (
    country_kpis
    .reset_index()
    .rename(columns={
        "line_total_gbp": "revenue_gbp",
        "invoice_no": "invoice_count",
        "customer_id": "identified_customer_count",
    })
)

country_kpis

,country,revenue_gbp,invoice_count,identified_customer_count
0,Belgium,1634.020,4,3
1,Channel Islands,838.750,1,1
2,Cyprus,76.520,1,1
3,Denmark,7870.600,2,2
4,EIRE,64232.050,31,5
...,...,...,...,...
14,Spain,4179.590,9,6
15,Sweden,1378.450,2,1
16,USA,2289.590,5,2
17,United Kingdom,502446.852,970,642


Triez `country_kpis` du CA le plus élevé au plus faible. Affichez les dix premiers pays.

In [29]:
# TODO A2
country_kpis.sort_values("revenue_gbp", ascending=False).head(10)

,country,revenue_gbp,invoice_count,identified_customer_count
17,United Kingdom,502446.852,970,642
4,EIRE,64232.050,31,5
11,Netherlands,29230.770,6,3
7,Germany,10806.370,20,12
6,France,8145.370,11,8
3,Denmark,7870.600,2,2
8,Greece,4798.350,3,2
14,Spain,4179.590,9,6
16,USA,2289.590,5,2
0,Belgium,1634.020,4,3


**A3. Analyse client uniquement.** Créez `identified_sales` en écartant les lignes sans identifiant client. Conservez toutes les lignes dans `sales` pour les autres analyses.

In [32]:
# TODO A3
identified_sales = sales.dropna(subset=["customer_id"]).copy()

print(f"lignes supp/sales : {sales.shape[0] - identified_sales.shape[0]}")    

lignes supp/sales : 8848


Calculez le CA par client identifié dans une table `customer_revenue`.

In [35]:
# TODO A3
customer_revenue = identified_sales.groupby("customer_id")["line_total_gbp"].sum()
customer_revenue

customer_id
12346     90.00
12361    109.20
12404     63.24
12406    881.20
12417    404.55
          ...  
18231    268.30
18242    326.54
18252    256.80
18259    306.30
18268    486.64
Name: line_total_gbp, Length: 696, dtype: float64

Créez et affichez `top_customers`, les cinq clients ayant généré le plus de CA (Series).

In [36]:
# TODO A3
top_customers = customer_revenue.sort_values(ascending=False).head()
top_customers.reset_index()

,customer_id,line_total_gbp
0,14156,51031.73
1,18102,38555.30
2,14646,28897.32
3,13694,19601.38
4,17949,11520.60


**Vérifiez vos résultats.** Une ligne par pays ; somme des CA pays égale au CA source ; cinq clients sans identifiant manquant.
Pour comparer des montants flottants, utilisez `np.isclose(a, b, atol=0.01)` plutôt que `a == b`.

In [37]:
## TEST YOUR CODE ##

# une ligne par pays
assert country_kpis["country"].is_unique
assert len(country_kpis) == sales["country"].nunique()

# Somme des CA = somme de toutes les commandes
assert np.isclose(
    country_kpis["revenue_gbp"].sum(), sales["line_total_gbp"].sum(),
    atol=0.01, # atol -> tolérance absolue de 1 centime
)

# top5
assert len(top_customers) == 5
assert top_customers.index.notna().all()

print("Contrôles A : OK")

Contrôles A : OK


**À discuter pendant la correction :** pourquoi retirer les clients inconnus dès le début ferait-il perdre du CA dans l'analyse par pays ?

## 2. Combiner des sources · 14h20–15h05
Nos indicateurs fonctionnent. Mais que faire si les ventes et les informations produits arrivent dans plusieurs fichiers ?

| Opération | Besoin | Question à se poser |
|---|---|---|
| `merge` | Enrichir des lignes avec une autre table | Quelle clé relie les deux tables ? |
| `concat` | Réunir des morceaux de même nature | Les colonnes décrivent-elles la même chose ? |

Démonstrations et échanges : **15 min** ; exercice B : **20 min** ; correction : **10 min**.

### Démonstration — Rattacher les informations du catalogue
**Avant :** les ventes contiennent un code livre ; le catalogue contient un titre, une catégorie et un éditeur.

In [38]:
bookstore_catalog = pd.DataFrame({
    "book_id": ["B1", "B2", "B3"],
    "title": ["Python au quotidien", "Cuisine facile", "Dessiner dehors"],
    "category": ["Tech", "Cuisine", "Loisirs"],
    "publisher": ["Éditions Adam", "Éditions Bob", "Éditions Adam"],
})

display(bookstore_sales)
display(bookstore_catalog)

,order_id,book_id,store,month,quantity,unit_price
0,J01,B1,Paris,janvier,2,20
1,J01,B2,Paris,janvier,1,15
2,J02,B1,Lyon,janvier,3,20
3,J03,B3,Lille,janvier,2,12
4,F01,B2,Lyon,février,4,15
5,F02,B3,Paris,février,1,12
6,F03,B1,Lille,février,2,22
7,F03,B3,Lille,février,3,12


,book_id,title,category,publisher
0,B1,Python au quotidien,Tech,Éditions Adam
1,B2,Cuisine facile,Cuisine,Éditions Bob
2,B3,Dessiner dehors,Loisirs,Éditions Adam


**Clé : `book_id`.** Plusieurs ventes peuvent concerner le même livre, mais le catalogue doit contenir une seule ligne par livre.
`how="left"` conserve toutes les lignes de ventes. `validate="many_to_one"` vérifie que la clé est unique à droite.

In [41]:
assert bookstore_catalog["book_id"].is_unique

bookstore_enriched = bookstore_sales.merge(
    bookstore_catalog,
    on="book_id",
    how="left",
    validate="many_to_one",
)

bookstore_enriched

,order_id,book_id,store,month,quantity,unit_price,title,category,publisher
0,J01,B1,Paris,janvier,2,20,Python au quotidien,Tech,Éditions Adam
1,J01,B2,Paris,janvier,1,15,Cuisine facile,Cuisine,Éditions Bob
2,J02,B1,Lyon,janvier,3,20,Python au quotidien,Tech,Éditions Adam
3,J03,B3,Lille,janvier,2,12,Dessiner dehors,Loisirs,Éditions Adam
4,F01,B2,Lyon,février,4,15,Cuisine facile,Cuisine,Éditions Bob
5,F02,B3,Paris,février,1,12,Dessiner dehors,Loisirs,Éditions Adam
6,F03,B1,Lille,février,2,22,Python au quotidien,Tech,Éditions Adam
7,F03,B3,Lille,février,3,12,Dessiner dehors,Loisirs,Éditions Adam


**Après :** les colonnes du catalogue sont rattachées à chaque vente.

In [40]:
assert len(bookstore_enriched) == len(bookstore_sales)
assert bookstore_enriched["title"].notna().all()

bookstore_enriched

,order_id,book_id,store,month,quantity,unit_price,title,category,publisher
0,J01,B1,Paris,janvier,2,20,Python au quotidien,Tech,Éditions Adam
1,J01,B2,Paris,janvier,1,15,Cuisine facile,Cuisine,Éditions Bob
2,J02,B1,Lyon,janvier,3,20,Python au quotidien,Tech,Éditions Adam
3,J03,B3,Lille,janvier,2,12,Dessiner dehors,Loisirs,Éditions Adam
4,F01,B2,Lyon,février,4,15,Cuisine facile,Cuisine,Éditions Bob
5,F02,B3,Paris,février,1,12,Dessiner dehors,Loisirs,Éditions Adam
6,F03,B1,Lille,février,2,22,Python au quotidien,Tech,Éditions Adam
7,F03,B3,Lille,février,3,12,Dessiner dehors,Loisirs,Éditions Adam


Une jointure qui s'exécute n'est pas forcément correcte : une clé dupliquée à droite peut multiplier les lignes, une clé absente peut laisser des informations manquantes.

### Démonstration — Réunir janvier et février
Les deux tables suivantes représentent des ventes de même nature, sur deux périodes différentes.

In [42]:
bookstore_january = bookstore_sales.loc[
    bookstore_sales["month"] == "janvier"
]
bookstore_february = bookstore_sales.loc[
    bookstore_sales["month"] == "février"
]

display(bookstore_january)
display(bookstore_february)

,order_id,book_id,store,month,quantity,unit_price
0,J01,B1,Paris,janvier,2,20
1,J01,B2,Paris,janvier,1,15
2,J02,B1,Lyon,janvier,3,20
3,J03,B3,Lille,janvier,2,12


,order_id,book_id,store,month,quantity,unit_price
4,F01,B2,Lyon,février,4,15
5,F02,B3,Paris,février,1,12
6,F03,B1,Lille,février,2,22
7,F03,B3,Lille,février,3,12


In [43]:
bookstore_combined = pd.concat(
    [bookstore_january, bookstore_february],
    ignore_index=True,
)

display(bookstore_combined)

,order_id,book_id,store,month,quantity,unit_price
0,J01,B1,Paris,janvier,2,20
1,J01,B2,Paris,janvier,1,15
2,J02,B1,Lyon,janvier,3,20
3,J03,B3,Lille,janvier,2,12
4,F01,B2,Lyon,février,4,15
5,F02,B3,Paris,février,1,12
6,F03,B1,Lille,février,2,22
7,F03,B3,Lille,février,3,12


`ignore_index=True` recrée un index continu. `concat` ne dédoublonne pas les lignes : deux fichiers qui se chevauchent resteraient un problème.

### Exercice B — Trois fichiers, une seule table exploitable
**20 minutes · Online Retail**

Les deux fichiers de ventes couvrent respectivement janvier–février et mars–avril.
Le catalogue apporte `product_name` et `product_family`. Le nom produit est la description la plus fréquente de son code (égalité départagée alphabétiquement).

Les familles sont ajoutées par des règles de mots-clés : Lighting, Kitchen, Storage, Jewellery, Decoration, puis Other. La première règle applicable gagne ; les détails figurent dans le script. Cette classification simplifiée n'est pas une nomenclature UCI.

In [ ]:
sales_part_1 = pd.read_csv(
    "sales_part_1.csv", dtype=identifier_types,
    parse_dates=["invoice_date"],
)
sales_part_2 = pd.read_csv(
    "sales_part_2.csv", dtype=identifier_types,
    parse_dates=["invoice_date"],
)

In [ ]:
products = pd.read_csv("products.csv", dtype={"stock_code": "string"})

display(sales_part_1.head(3))
display(sales_part_2.head(3))
display(products.sample(3))

**B1. Reconstruire.** Choisissez l'opération adaptée pour réunir les deux périodes dans `sales_combined`, avec un index continu.

In [ ]:
# TODO B1


Vérifiez le nombre total de lignes et la reconstruction exacte de `sales_source` (table d'origine non enrichie).
Les fichiers sont déjà triés chronologiquement. `pd.testing.assert_frame_equal(table_1, table_2)` vérifie valeurs, colonnes, types et ordre.

In [ ]:
# TODO


**B2. Enrichir.** Identifiez la clé commune entre ventes et catalogue. Vérifiez qu'elle est unique dans le catalogue avant de poursuivre.

In [ ]:
# TODO B2


Construisez `sales_enriched` avec les informations produit, en conservant toutes les ventes. Faites vérifier par Pandas que plusieurs ventes peuvent correspondre à un seul produit du catalogue.

In [ ]:
# TODO


Contrôlez le nombre de lignes avant/après et l'absence de ventes sans informations produit. Affichez cinq lignes enrichies.

In [ ]:
# TEST YOUR CODE
assert len(sales_enriched) == len(sales_combined)
assert sales_enriched[["product_name", "product_family"]].notna().all().all()
display(sales_enriched.head())
print("Contrôles B : OK")

**B3. Expliquez en commentaires :** pourquoi `merge` serait-il inadapté pour reconstruire les deux périodes ? Pourquoi `concat` serait-il inadapté pour ajouter les familles produit ?

In [ ]:
# TODO B3


## Pause · 15h05–15h20
Reprise à 15h20 : mêmes données, autre forme.

## 3. Changer la forme des données · 15h20–16h00
**Table longue → pivot → table large → melt → table longue**

En format long, une dimension comme le magasin est une colonne dont les valeurs se répètent.
En format large, chaque magasin devient une colonne.

`pivot_table` peut **agréger** plusieurs lignes dans une cellule. `melt` remet les colonnes en lignes,
mais ne recrée pas les transactions individuelles perdues dans l'agrégation.

Démonstration : **10 min** ; exercice C : **18 min** ; correction et consolidation : **12 min**.

### Démonstration — AVANT : les quantités vendues en format long

In [ ]:
bookstore_long = bookstore_enriched[["category", "store", "quantity"]]
bookstore_long

### PIVOT : une catégorie par ligne, un magasin par colonne
`index` fixe les lignes ; `columns` fixe les colonnes ; `values` choisit la mesure ; `aggfunc` choisit le calcul.
Nous écrivons explicitement `aggfunc="sum"` : sans cela, la valeur par défaut serait une moyenne.
Ici, `fill_value=0` signifie « aucune vente observée pour cette combinaison ».

In [ ]:
bookstore_wide = bookstore_long.pivot_table(
    index="category",
    columns="store",
    values="quantity",
    aggfunc="sum",
    fill_value=0,
)

bookstore_wide

### MELT : retrouver catégorie, magasin et quantité
On remet d'abord la catégorie dans une colonne.

In [ ]:
bookstore_wide_flat = bookstore_wide.reset_index()
bookstore_wide_flat

In [ ]:
bookstore_melted = bookstore_wide_flat.melt(
    id_vars="category",
    var_name="store",
    value_name="quantity",
)

bookstore_melted

`id_vars` désigne les colonnes conservées comme identifiants. Les autres colonnes deviennent des valeurs de `store`.
La table obtenue contient **une ligne par combinaison catégorie–magasin**, avec des quantités déjà additionnées.

### Exercice C — Préparer une vue temporelle pour l'analyse
**18 minutes · Online Retail · table `sales` de l'exercice A**

L'équipe souhaite comparer le CA des marchés au fil des quatre mois.
Le starter fournit le mois ; votre travail porte sur la forme des tables.

In [ ]:
sales["invoice_date"] = pd.to_datetime(sales["invoice_date"])

In [ ]:
sales["invoice_month"] = sales["invoice_date"].dt.strftime("%Y-%m")
display(sales[["invoice_date", "invoice_month"]].head(3))

**C1. Vue large.** Créez `monthly_market_revenue` : une ligne par mois, une colonne par marché, le CA total dans chaque cellule. Une combinaison sans vente doit afficher zéro. Choisissez les paramètres adaptés et affichez toute la table.

In [ ]:
# TODO C1

**C2. Préparer le retour au long.** Remettez le mois dans une colonne de `monthly_market_flat`.

In [ ]:
# TODO C2

Créez `monthly_market_long` avec exactement `invoice_month`, `market`, `revenue_gbp`. Affichez cette table.

In [ ]:
# TODO

**Contrôler.** Vérifiez la présence de quatre mois et des marchés attendus. Vérifiez numériquement : CA source = somme de la vue large = somme de la vue longue.
Pour additionner toutes les cellules numériques d'un pivot : `table.to_numpy().sum()`.

In [ ]:
# TEST YOUR CODE

assert monthly_market_revenue.shape == (4, sales["market"].nunique())

source_revenue = sales["line_total_gbp"].sum()
assert np.isclose(
    monthly_market_revenue.to_numpy().sum(), source_revenue,
    atol=0.01,
)

assert np.isclose(
    monthly_market_long["revenue_gbp"].sum(), source_revenue,
    atol=0.01,
)

print("Contrôles C : OK")

## 5. TP final — Préparer le brief commercial · 16h20–17h00
**28 minutes de travail · 12 minutes de correction et consolidation**

La direction souhaite comparer **les marchés** et **les familles de produits**.
Préparez trois tables fiables, réutilisables pour la visualisation de la prochaine séance.
Travaillez à partir de `sales_analysis.csv` et `products.csv`. Les tables de départ sont rechargées ci-dessous : le TP ne dépend pas de vos résultats précédents.

| Livrable | Forme attendue |
|---|---|
| `market_kpis` | Une ligne par marché ; `market`, `revenue_gbp`, `invoice_count`, `identified_customer_count`, `average_revenue_per_invoice` |
| `market_family_revenue` | Une ligne par couple marché–famille ; `market`, `product_family`, `revenue_gbp` |
| `market_family_matrix` | Une famille par ligne, un marché par colonne ; CA dans les cellules |

`average_revenue_per_invoice` = CA du marché / nombre de factures distinctes de ce marché.
Les clients inconnus restent inclus dans le CA et les factures ; ils ne sont pas comptés comme clients identifiés.
Les informations du catalogue doivent être rattachées aux transactions avant d'analyser les familles.
Choisissez vous-mêmes les opérations et l'ordre de travail.

In [ ]:
brief_sales = pd.read_csv(
    "sales_analysis.csv", dtype=identifier_types,
    parse_dates=["invoice_date"],
)
brief_products = pd.read_csv("products.csv", dtype={"stock_code": "string"})

### Espace de travail — Préparation
Utilisez les cellules suivantes pour séparer vos étapes. Vous pouvez en ajouter si nécessaire.

Préparez la mesure nécessaire à votre analyse.

In [ ]:
# TODO

### Livrable 1 — `market_kpis`
La table doit permettre de comparer le CA, le nombre de factures, les clients identifiés et le CA moyen par facture.

Construisez vos indicateurs par marché.

In [ ]:
# TODO

Donnez à la table les colonnes demandées.

In [ ]:
# TODO

Complétez l'indicateur moyen et affichez le livrable.

In [ ]:
# TODO

### Livrable 2 — `market_family_revenue`
Rattachez les informations produit aux transactions, puis préparez la synthèse marché–famille.
Votre démarche doit démontrer que les ventes n'ont été ni perdues ni multipliées.

Rattachez le catalogue aux ventes dans une table de travail.

In [ ]:
# TODO

Contrôlez le rattachement avant de calculer les indicateurs.

In [ ]:
# TODO

Construisez la synthèse demandée.

In [ ]:
# TODO

Finalisez les noms des colonnes et affichez un aperçu.

In [ ]:
# TODO

### Livrable 3 — `market_family_matrix`
Proposez la vue large qui permettra de comparer directement les familles entre les marchés.

Créez et affichez la matrice complète ; les combinaisons sans ventes doivent contenir zéro.

In [ ]:
# TODO

### Contrôles de livraison
Avant de remettre le brief, vérifiez :

- une ligne par marché dans les KPI ; plusieurs marchés et plusieurs familles présents ;
- aucune transaction perdue ou multipliée, aucun produit non rattaché ;
- le même CA total dans la source, les KPI, la synthèse marché–famille et la matrice ;
- une matrice contenant des valeurs pour la plupart des couples marché–famille.

Séparez les vérifications structurelles et les vérifications de montants.

Vérifiez la structure et la couverture de vos résultats.

In [ ]:
# TODO

Vérifiez la conservation du CA entre les quatre tables.

In [ ]:
# TODO

### Consolidation orale
Présentez un constat chiffré tiré de vos tables, puis expliquez un contrôle qui vous donne confiance dans ce constat.
Conservez votre notebook dans Colab : les tables pourront servir à la prochaine séance.